# Michigan 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Michigan, 2008 for presidential general election results, and then derive summary stats (party totals). Note, there is no presidential primary election results for Arkansas 2008 so far.

**Output**: A single CSV where each row is a county and columns include:

- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_general_total`, `dem_general_total`, `lib_general_total`, `grn_general_total`, `nlp_general_total`,`ust_general_total`, `npa_general_total`

**Last Updated**: 2025/10/11

## 0. Library Import

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

Define raw file paths once here so the entire notebook is easy to rerun on another machine. If a path changes, we only update it here. We keep a single `OUTPUT_PATH` so all exports land in one known place.

In [2]:
# MI 2008 dataset path
# PRIMARY_PATH = r""
GENERAL_PATH = r"../../data/raw/2008/MI/20081104__mi__general__precinct.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/MI/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### b. General Election Dataset

In [3]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,votes
0,Alcona,Alcona Township 1,President,NaN,NPA,Brian Moore,0
1,Alcona,Alcona Township 1,President,NaN,NPA,Alan Keyes,0
2,Alcona,Alcona Township 1,President,NaN,DEM,Barack Obama,329
3,Alcona,Alcona Township 1,President,NaN,REP,John Mccain,423
4,Alcona,Alcona Township 1,President,NaN,NLP,Ralph Nader,8
5,Alcona,Alcona Township 1,President,NaN,GRN,Cynthia Mckinney,4
6,Alcona,Alcona Township 1,President,NaN,LIB,Bob Barr,2
7,Alcona,Alcona Township 1,President,NaN,UST,Chuck Baldwin,1
8,Alcona,Alcona Township 1,U.S. Senate,NaN,NLP,Doug Dern,1
9,Alcona,Alcona Township 1,U.S. Senate,NaN,GRN,Harley G. Mikkelson,6


In [4]:
# Different values in 'office' column
general_df["office"].value_counts()

office
President      45656
U.S. Senate    34128
U.S. House     25581
State House    13816
Name: count, dtype: int64

In [5]:
# Only keep rows where 'office' is 'President'
general_df = general_df[general_df["office"] == "President"]
general_df.shape

(45656, 7)

In [7]:
# Number of missing values in 'district' column
general_df["district"].isna().sum()

45656

Here, we can see there are no values in `district` column. Thus, we can drop both `office` and `district` columns since they don't provide us with any additional information now.

In [8]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the district column as it's not applicable 
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,precinct,party,candidate,votes
0,Alcona,Alcona Township 1,NPA,Brian Moore,0
1,Alcona,Alcona Township 1,NPA,Alan Keyes,0
2,Alcona,Alcona Township 1,DEM,Barack Obama,329
3,Alcona,Alcona Township 1,REP,John Mccain,423
4,Alcona,Alcona Township 1,NLP,Ralph Nader,8
5,Alcona,Alcona Township 1,GRN,Cynthia Mckinney,4
6,Alcona,Alcona Township 1,LIB,Bob Barr,2
7,Alcona,Alcona Township 1,UST,Chuck Baldwin,1
8,Alcona,Caledonia Township 1,NPA,Brian Moore,0
9,Alcona,Caledonia Township 1,NPA,Alan Keyes,0


Now, we aggregate precinct vote counts into county vote counts.

In [9]:
# Make sure votes are numeric
general_df["votes"] = pd.to_numeric(general_df["votes"], errors="coerce").fillna(0).astype(int)

# Aggregate precinct vote counts into county vote counts
general_df = (
    general_df.
    groupby(["county", "party", "candidate"], as_index=False)["votes"]
    .sum()
)[["county", "candidate", "party", "votes"]]        # Reorder columns

# Snippet at the aggregated data
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Alcona,Barack Obama,DEM,2896
1,Alcona,Cynthia Mckinney,GRN,18
2,Alcona,Bob Barr,LIB,36
3,Alcona,Ralph Nader,NLP,51
4,Alcona,Alan Keyes,NPA,0
5,Alcona,Brian Moore,NPA,0
6,Alcona,John Mccain,REP,3404
7,Alcona,Chuck Baldwin,UST,15
8,Alger,Barack Obama,DEM,2472
9,Alger,Cynthia Mckinney,GRN,14


In [10]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
NPA    166
DEM     83
GRN     83
LIB     83
NLP     83
REP     83
UST     83
Name: count, dtype: int64

In [11]:
# Missing values count
general_df.isnull().sum()

county       0
candidate    0
party        0
votes        0
dtype: int64

In [12]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Alcona,Barack Obama,DEM,2896
1,Alcona,Cynthia Mckinney,GRN,18
2,Alcona,Bob Barr,LIB,36
3,Alcona,Ralph Nader,NLP,51
4,Alcona,Alan Keyes,NPA,0
5,Alcona,Brian Moore,NPA,0
6,Alcona,John Mccain,REP,3404
7,Alcona,Chuck Baldwin,UST,15
8,Alger,Barack Obama,DEM,2472
9,Alger,Cynthia Mckinney,GRN,14


In [13]:
# Shape after preprocessing
general_df.shape

(664, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [14]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Lowering the three-letter abbreviations
    """
    return(s.str.strip().str.lower())

In [15]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [16]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [17]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_nlp_NADER,gen_npa_KEYES,gen_npa_MOORE,gen_rep_MCCAIN,gen_ust_BALDWIN
0,Alcona,2896,18,36,51,0,0,3404,15
1,Alger,2472,14,19,44,0,0,2188,13
2,Allegan,24165,109,294,385,0,0,30061,265
3,Alpena,7705,25,79,121,0,0,7125,30
4,Antrim,6079,40,68,104,0,0,7506,55
5,Arenac,4155,22,36,86,0,0,3807,22
6,Baraga,1725,4,18,31,0,0,1846,20
7,Barry,13449,78,189,243,0,0,16431,175
8,Bay,32589,100,254,510,0,0,23795,180
9,Benzie,5451,21,36,66,0,0,4687,48


In [18]:
# General dataframe shape after pivot
general_pivot.shape

(83, 9)

## 4. Adding Party Total Columns

Now, we will add party totals columns for general totals:

* `rep_general_total` = sum of all `gen_rep_*` columns
* `dem_general_total` = sum of all `gen_dem_*` columns
* `lib_general_total` = sum of all `gen_lib_*` columns
* `grn_general_total` = sum of all `gen_grn_*` columns
* `nlp_general_total` = sum of all `gen_nlp_*` columns
* `ust_general_total` = sum of all `gen_ust_*` columns
* `npa_general_total` = sum of all `gen_npa_*` columns

In [19]:
# Add party totals for general election
rep_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_rep")] 
dem_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_dem")]
lib_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_lib")]
grn_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_grn")]
nlp_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_nlp")]
ust_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_ust")]
npa_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_npa")]

general_pivot["rep_general_total"] = general_pivot[rep_general_cols].sum(axis=1) if rep_general_cols else 0
general_pivot["dem_general_total"] = general_pivot[dem_general_cols].sum(axis=1) if dem_general_cols else 0
general_pivot["lib_general_total"] = general_pivot[lib_general_cols].sum(axis=1) if lib_general_cols else 0
general_pivot["grn_general_total"] = general_pivot[grn_general_cols].sum(axis=1) if grn_general_cols else 0
general_pivot["nlp_general_total"] = general_pivot[nlp_general_cols].sum(axis=1) if nlp_general_cols else 0
general_pivot["ust_general_total"] = general_pivot[ust_general_cols].sum(axis=1) if ust_general_cols else 0
general_pivot["npa_general_total"] = general_pivot[npa_general_cols].sum(axis=1) if npa_general_cols else 0


In [21]:
# Print out all the column names in the final dataframe
general_pivot.columns

Index(['county', 'gen_dem_OBAMA', 'gen_grn_MCKINNEY', 'gen_lib_BARR',
       'gen_nlp_NADER', 'gen_npa_KEYES', 'gen_npa_MOORE', 'gen_rep_MCCAIN',
       'gen_ust_BALDWIN', 'rep_general_total', 'dem_general_total',
       'lib_general_total', 'grn_general_total', 'nlp_general_total',
       'ust_general_total', 'npa_general_total'],
      dtype='object')

In [22]:
# Preview the general_pivot dataframe with totals
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_nlp_NADER,gen_npa_KEYES,gen_npa_MOORE,gen_rep_MCCAIN,gen_ust_BALDWIN,rep_general_total,dem_general_total,lib_general_total,grn_general_total,nlp_general_total,ust_general_total,npa_general_total
0,Alcona,2896,18,36,51,0,0,3404,15,3404,2896,36,18,51,15,0
1,Alger,2472,14,19,44,0,0,2188,13,2188,2472,19,14,44,13,0
2,Allegan,24165,109,294,385,0,0,30061,265,30061,24165,294,109,385,265,0
3,Alpena,7705,25,79,121,0,0,7125,30,7125,7705,79,25,121,30,0
4,Antrim,6079,40,68,104,0,0,7506,55,7506,6079,68,40,104,55,0
5,Arenac,4155,22,36,86,0,0,3807,22,3807,4155,36,22,86,22,0
6,Baraga,1725,4,18,31,0,0,1846,20,1846,1725,18,4,31,20,0
7,Barry,13449,78,189,243,0,0,16431,175,16431,13449,189,78,243,175,0
8,Bay,32589,100,254,510,0,0,23795,180,23795,32589,254,100,510,180,0
9,Benzie,5451,21,36,66,0,0,4687,48,4687,5451,36,21,66,48,0


Now, we save the cleaned dataframe into the processed directory.

In [23]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
general_pivot.to_csv(OUTPUT_PATH + "MI.csv", index=False)